<a href="https://colab.research.google.com/github/LAtifword/latif-brain/blob/latif-brain/training/LATIF_NI_Train_LoRA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LATIF NI — real LoRA training on a free Colab GPU
Run in Google Colab with **Runtime → Change runtime type → T4 GPU** (if available). This trains **Qwen2.5-0.5B**, **not Gemini Nano**. No GPU is guaranteed; do not upload confidential datasets you lack permission to process in Colab.
Prepare a UTF-8 JSONL file with **at least 20 unique examples**: one `{ "text": "..." }` per line, each text at least 30 characters. Keep a copy offline.

In [7]:
import torch
assert torch.cuda.is_available(), 'No CUDA GPU assigned. Choose GPU runtime and reconnect.'
print('CUDA:', torch.cuda.get_device_name(0))
!git clone --depth 1 --branch latif-brain https://github.com/LAtifword/latif-brain.git /content/latif-brain
%cd /content/latif-brain
!python -m pip install -q -r training/requirements.txt 'datasets>=3,<5' 'torchao>=0.16.0'

CUDA: Tesla T4
fatal: destination path '/content/latif-brain' already exists and is not an empty directory.
/content/latif-brain


In [9]:
from google.colab import files
from pathlib import Path
uploaded = files.upload()
assert len(uploaded)==1, 'Upload exactly one training JSONL file'
name = next(iter(uploaded))
Path('/content/latif-train.jsonl').write_bytes(uploaded[name])
!python training/validate_dataset.py --input /content/latif-train.jsonl --output training/data
del uploaded


KeyboardInterrupt: 

In [ ]:
from pathlib import Path
from google.colab import drive
import shutil

drive.mount("/content/drive")

src = Path("/content/latif-brain/artifacts/lora")
dst = Path("/content/drive/MyDrive/LATIF-NI/models/lora-v1")

assert (src / "adapter_model.safetensors").is_file(), \
    f"Adapter missing: {src}"

    dst.mkdir(parents=True, exist_ok=True)

    shutil.copytree(src, dst, dirs_exist_ok=True)

    print("BACKUP SUCCESSFUL:", dst)

    for p in dst.rglob("*"):
        if p.is_file():
                print(p.name, p.stat().st_size, "bytes")

In [11]:
from pathlib import Path
from google.colab import drive
import shutil

drive.mount("/content/drive")

src = Path("/content/latif-brain/artifacts/lora")
dst = Path("/content/drive/MyDrive/LATIF-NI/models/lora-v1")

assert (src / "adapter_model.safetensors").is_file(), \
    "Training adapter not found"

    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(src, dst, dirs_exist_ok=True)

    print("Saved trained adapter:", dst)
    print("Files:")
    for p in dst.rglob("*"):
        if p.is_file():
                print(p.name, p.stat().st_size, "bytes")

IndentationError: unexpected indent (4008135071.py, line 13)

In [8]:
import subprocess
from pathlib import Path

model = 'Qwen/Qwen2.5-0.5B-Instruct'

print("Starting real GPU training run and capturing output...")
result = subprocess.run([
    'python', 'training/train_lora.py',
    '--model', model,
    '--data', 'training/data/train.jsonl',
    '--output', 'artifacts/lora',
    '--steps', '100',
    '--max-length', '256'
], capture_output=True, text=True)

print("--- STDOUT ---")
print(result.stdout)
print("--- STDERR ---")
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(f"Training failed with exit code {result.returncode}")

adapter_path = Path('artifacts/lora/adapter_model.safetensors')
assert adapter_path.is_file(), 'Adapter file missing: Training failed to save weights.'
print('Training completed successfully! Adapter weight file exists at:', adapter_path)

Starting real GPU training run and capturing output...
--- STDOUT ---
trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093
{'loss': 2.2713, 'grad_norm': 0.7762365341186523, 'learning_rate': 0.000182, 'epoch': 0.22}
{'loss': 1.9216, 'grad_norm': 0.744179368019104, 'learning_rate': 0.000162, 'epoch': 0.43}
{'loss': 1.9236, 'grad_norm': 0.919496476650238, 'learning_rate': 0.000142, 'epoch': 0.65}
{'loss': 1.9833, 'grad_norm': 1.3686962127685547, 'learning_rate': 0.000122, 'epoch': 0.87}
{'loss': 1.8931, 'grad_norm': 0.840906023979187, 'learning_rate': 0.00010200000000000001, 'epoch': 1.07}
{'loss': 1.7149, 'grad_norm': 0.9846086502075195, 'learning_rate': 8.2e-05, 'epoch': 1.28}
{'loss': 1.7984, 'grad_norm': 1.1799958944320679, 'learning_rate': 6.2e-05, 'epoch': 1.5}
{'loss': 1.6624, 'grad_norm': 0.8100718259811401, 'learning_rate': 4.2e-05, 'epoch': 1.72}
{'loss': 1.8529, 'grad_norm': 1.1761912107467651, 'learning_rate': 2.2000000000000003e-05, 'epoch': 1.93}
{'loss

In [10]:
import subprocess
from pathlib import Path

model = 'Qwen/Qwen2.5-0.5B-Instruct'

print("Evaluating the trained model on held-out validation set...")
result = subprocess.run([
    'python', 'training/evaluate_lora.py',
    '--model', model,
    '--adapter', 'artifacts/lora',
    '--data', 'training/data/validation.jsonl'
], capture_output=True, text=True)

print("--- EVALUATION OUTPUT ---")
print(result.stdout)
print(result.stderr)

eval_json = Path('artifacts/evaluation.json')
if eval_json.is_file():
    print("\nSuccessfully saved evaluation metrics to:", eval_json)
    print(eval_json.read_text())
else:
    raise RuntimeError("Evaluation failed: Output metrics file not found.")

Evaluating the trained model on held-out validation set...
--- EVALUATION OUTPUT ---
{"examples": 40, "tokens": 12997, "mean_token_nll": 1.454040614808533, "perplexity": 4.280374966743743, "base_model": "Qwen/Qwen2.5-0.5B-Instruct", "adapter": "artifacts/lora"}

I0000 00:00:1792914059.923270    8480 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792914062.334078    8480 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
`torch_dtype` is deprecated! Use `dtype` instead!


Successfully saved evaluation metrics to: artifacts/evaluation.json
{
  "examples": 40,
  "tokens": 12997,
  "mean_token_nll": 1.454040614808533,
  "perplexity": 4.280374966743743,
  "base_model": "Qwen/Qwen2.5-0.5B-Instruct",
  "adapter": "artifacts/lora"
}

In [12]:
import zipfile
from pathlib import Path
from google.colab import files

archive = '/content/LATIF-NI-QWEN-LORA.zip'
print("Creating ZIP archive containing the LoRA adapter weights and evaluation files...")

with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    # Compress LoRA weights and configs
    lora_dir = Path('artifacts/lora')
    if lora_dir.is_dir():
        for p in lora_dir.rglob('*'):
            if p.is_file():
                z.write(p, p.relative_to(Path('artifacts').parent).as_posix())
                print(f"Added: {p.as_posix()}")

    # Compress evaluation report
    eval_file = Path('artifacts/evaluation.json')
    if eval_file.is_file():
        z.write(eval_file, eval_file.as_posix())
        print(f"Added: {eval_file.as_posix()}")

    # Compress dataset manifest
    manifest_file = Path('training/data/manifest.json')
    if manifest_file.is_file():
        z.write(manifest_file, manifest_file.as_posix())
        print(f"Added: {manifest_file.as_posix()}")

print("\nArchive ready. Initiating browser download...")
files.download(archive)
print('Download initiated! This package contains the adapter weights and evaluation json.')

Creating ZIP archive containing the LoRA adapter weights and evaluation files...
Added: artifacts/lora/added_tokens.json
Added: artifacts/lora/vocab.json
Added: artifacts/lora/tokenizer.json
Added: artifacts/lora/merges.txt
Added: artifacts/lora/tokenizer_config.json
Added: artifacts/lora/README.md
Added: artifacts/lora/chat_template.jinja
Added: artifacts/lora/special_tokens_map.json
Added: artifacts/lora/adapter_model.safetensors
Added: artifacts/lora/adapter_config.json
Added: artifacts/lora/checkpoint-50/scheduler.pt
Added: artifacts/lora/checkpoint-50/added_tokens.json
Added: artifacts/lora/checkpoint-50/vocab.json
Added: artifacts/lora/checkpoint-50/rng_state.pth
Added: artifacts/lora/checkpoint-50/tokenizer.json
Added: artifacts/lora/checkpoint-50/merges.txt
Added: artifacts/lora/checkpoint-50/tokenizer_config.json
Added: artifacts/lora/checkpoint-50/training_args.bin
Added: artifacts/lora/checkpoint-50/README.md
Added: artifacts/lora/checkpoint-50/scaler.pt
Added: artifacts/lor

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download initiated! This package contains the adapter weights and evaluation json.
